In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.datasets import mroz

from pymargins import Margins

cols = ["inlf", "nwifeinc", "educ", "exper", "age", "kidslt6", "kidsge6"]
df = mroz.load()[cols].dropna().copy()
df["expersq"] = df["exper"] ** 2

fit = smf.glm(
    "inlf ~ nwifeinc + educ + exper + expersq + age + kidslt6 + kidsge6",
    data=df,
    family=sm.families.Binomial(),
).fit()
print(fit.summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4255      0.860      0.495      0.621      -1.261       2.112
nwifeinc      -0.0213      0.008     -2.535      0.011      -0.038      -0.005
educ           0.2212      0.043      5.091      0.000       0.136       0.306
exper          0.2059      0.032      6.422      0.000       0.143       0.269
expersq       -0.0032      0.001     -3.104      0.002      -0.005      -0.001
age           -0.0880      0.015     -6.040      0.000      -0.117      -0.059
kidslt6       -1.4434      0.204     -7.090      0.000      -1.842      -1.044
kidsge6        0.0601      0.075      0.804      0.422      -0.086       0.207


In [2]:
beta_hat = np.asarray(fit.params)
V_hat = np.asarray(fit.cov_params())

rng = np.random.default_rng(0)
posterior_draws = rng.multivariate_normal(beta_hat, V_hat, size=4000)
print("draw bank shape:", posterior_draws.shape)  # (n_draws, n_params)

draw bank shape: (4000, 8)


In [3]:
m_bayes = Margins.from_posterior(fit, posterior_draws, at="overall")
print("method:", m_bayes.method, " draws:", m_bayes.n_sim)

method: simulation  draws: 4000


In [4]:
m_delta = Margins.linear_scale(fit, at="overall")

ame_bayes = m_bayes.dydx("educ")
ame_delta = m_delta.dydx("educ")

print("Posterior (credible interval):")
print(ame_bayes.summary())
print("\nDelta method (confidence interval):")
print(ame_delta.summary())

Posterior (credible interval):
          Margins Result (simulation, level=0.95)          
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.0395   0.0072     0.0395  0.000    0.0247, 0.0528

n = 753
κ: 0.055

Delta method (confidence interval):
           Margins Result (delta, level=0.95)           
      estimate  std err       z  P>|z|  [95% Conf. Int.]
--------------------------------------------------------
educ    0.0395   0.0073  5.4144  0.000    0.0252, 0.0538

n = 753
κ: 0.054
Delta-vs-sim disagreement: 4.007%


In [5]:
from pymargins import pairwise

scen, w = pairwise("kidslt6", [1, 0])
child_effect = m_bayes.contrasts(scenarios=scen, contrasts=w)
print(child_effect.summary())

draws = np.asarray(child_effect.draws)
print(f"\nPosterior mean : {draws.mean():.4f}")
print(f"P(effect < 0)  : {(draws < 0).mean():.3f}")
print(f"90% credible   : [{np.percentile(draws, 5):.4f}, "
      f"{np.percentile(draws, 95):.4f}]")

            Margins Result (simulation, level=0.95)             
           estimate  std err  statistic  P>|z|  [95% Conf. Int.]
----------------------------------------------------------------
kidslt6=1   -0.2697   0.0347    -0.2697  0.000  -0.3321, -0.1982

n = 753
κ: 0.057

Posterior mean : -0.2668
P(effect < 0)  : 1.000
90% credible   : [-0.3228, -0.2100]
